# Day 2 — Airport Operations AI Copilot

## Operational Data, Tools & Gemini Function Calling

### Objective

Give the AI access to mock airport operational data and teach it to
select and execute tools when additional information is required.

### Day 2 Workflow

User Query
→ Gemini
→ Function Call
→ Python Tool
→ Tool Result
→ Gemini
→ Final Answer

In [1]:
import pandas as pd

df = pd.read_csv("../data/airport_metrics.csv")

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (24, 9)


,airport_code,completion_rate,average_eta,active_drivers,driver_cancellation_rate,queue_size,surge_multiplier,request_volume,timestamp
0,SFO,0.91,8.2,420,0.06,185,1.2,3850,2026-09-15 09:00:00
1,SFO,0.89,9.1,395,0.08,210,1.3,4100,2026-09-15 10:00:00
2,SFO,0.87,10.4,370,0.10,245,1.4,4350,2026-09-15 11:00:00
3,SFO,0.90,9.0,405,0.07,220,1.3,4200,2026-09-15 12:00:00
4,SFO,0.93,7.5,440,0.05,170,1.1,3600,2026-09-15 13:00:00


In [2]:
print("Airports:")
print(sorted(df["airport_code"].unique()))

print("\nRecords per airport:")
print(df["airport_code"].value_counts())

Airports:
['JFK', 'LAX', 'SFO']

Records per airport:
airport_code
SFO    8
LAX    8
JFK    8
Name: count, dtype: int64


In [3]:
print("Missing values:")
print(df.isnull().sum())

Missing values:
airport_code                0
completion_rate             0
average_eta                 0
active_drivers              0
driver_cancellation_rate    0
queue_size                  0
surge_multiplier            0
request_volume              0
timestamp                   0
dtype: int64


## 2. Data Preprocessing

Before operational data is used by the AI tools, it is validated
through the preprocessing layer.

The preprocessing function checks:

1. Required columns are present
2. Airport codes are valid
3. Timestamp values can be parsed
4. Numeric fields contain valid numeric values
5. No required values are missing

This creates a clean and validated dataset for the tool layer.

In [8]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)



Project root: /home/nineleaps/Documents/airport-ai-copilot


In [5]:
from src.data_preprocessing import load_airport_metrics

processed_df = load_airport_metrics(
    "../data/airport_metrics.csv"
)

print("Rows:", len(processed_df))
print("Columns:", list(processed_df.columns))
print("Airports:", sorted(processed_df["airport_code"].unique()))

Rows: 24
Columns: ['airport_code', 'completion_rate', 'average_eta', 'active_drivers', 'driver_cancellation_rate', 'queue_size', 'surge_multiplier', 'request_volume', 'timestamp']
Airports: ['JFK', 'LAX', 'SFO']


## 3. Tool 1 — get_airport_metrics()

This tool retrieves the latest operational metrics for a given airport.

Input:
- airport_code

Output:
- completion rate
- average ETA
- active drivers
- driver cancellation rate
- queue size
- surge multiplier
- request volume
- timestamp

The tool retrieves the latest available record for the requested airport.

In [15]:
from src.tools import get_airport_metrics

sfo_metrics = get_airport_metrics("SFO")

sfo_metrics

{'status': 'success',
 'airport_code': 'SFO',
 'completion_rate': 0.86,
 'average_eta': 11.2,
 'active_drivers': 355,
 'driver_cancellation_rate': 0.11,
 'queue_size': 260,
 'surge_multiplier': 1.4,
 'request_volume': 4500,
 'timestamp': '2026-09-15 16:00:00'}

## 4. Tool 2 — calculate_driver_incentive()

This tool calculates the recommended incentive per driver and
the estimated total incentive cost.

Inputs:
- driver_count
- severity_level

Synthetic project rules:

- low → $10 per driver
- medium → $20 per driver
- high → $35 per driver

In [16]:
from src.tools import calculate_driver_incentive

incentive_result = calculate_driver_incentive(
    driver_count=100,
    severity_level="high"
)

incentive_result

{'status': 'success',
 'driver_count': 100,
 'severity_level': 'high',
 'recommended_incentive_per_driver': 35,
 'estimated_total_cost': 3500}

## 5. Tool 3 — trigger_surge_override()

This tool simulates changing the surge multiplier for an airport.

Inputs:
- airport_code
- new_multiplier
- reason

This is a mock execution tool for Day 2.

Actual approval and permission controls will be added on Day 4.

In [17]:
from src.tools import trigger_surge_override

surge_result = trigger_surge_override(
    airport_code="SFO",
    new_multiplier=1.4,
    reason="High queue size and elevated request volume"
)

surge_result

{'status': 'success',
 'action': 'surge_override',
 'airport_code': 'SFO',
 'new_multiplier': 1.4,
 'reason': 'High queue size and elevated request volume',
 'execution_mode': 'mock',
 'message': 'Mock surge override executed for SFO at 1.4x.'}

## 6. Tool Registry

The tool registry provides a structured description of the tools
available to the AI assistant.

For each tool, the registry defines:

- Tool name
- Description
- Required parameters
- Parameter types

The LLM uses these definitions to decide which tool should be
called and what arguments should be provided.

In [18]:
from src.tools import TOOL_REGISTRY

TOOL_REGISTRY

for tool_name, tool_info in TOOL_REGISTRY.items():
    print("=" * 50)
    print("Tool:", tool_name)
    print("Description:", tool_info["description"])
    print("Parameters:")

    for parameter_name, parameter_info in tool_info["parameters"].items():
        print(
            f"  - {parameter_name}: "
            f"{parameter_info['type']} | "
            f"required={parameter_info['required']}"
        )

Tool: get_airport_metrics
Description: Retrieve the latest operational metrics for an airport, including completion rate, ETA, active drivers, driver cancellation rate, queue size, surge multiplier, request volume, and timestamp.
Parameters:
  - airport_code: string | required=True
Tool: calculate_driver_incentive
Description: Calculate the recommended incentive per driver and estimated total incentive cost based on driver count and operational severity.
Parameters:
  - driver_count: integer | required=True
  - severity_level: string | required=True
Tool: trigger_surge_override
Description: Mock execution tool that simulates changing the surge multiplier for an airport.
Parameters:
  - airport_code: string | required=True
  - new_multiplier: number | required=True
  - reason: string | required=True


## 7. Gemini Function Calling

The Gemini model will receive the available tool definitions.

The model can decide:

1. Whether a tool is required
2. Which tool should be called
3. What arguments should be passed

The Python application then executes the selected tool and sends
the tool result back to Gemini.

Flow:

User Query
    ↓
Gemini
    ↓
Tool Selection
    ↓
Python Tool Execution
    ↓
Tool Result
    ↓
Gemini
    ↓
Final Answer

In [19]:
from src.gemini_tools import create_gemini_client, create_gemini_tools

client = create_gemini_client()
tools = create_gemini_tools()

print("Gemini client created.")
print("Tools configured:", len(tools))

Gemini client created.
Tools configured: 1


In [20]:
from google.genai import types

query = "What's happening at SFO?"

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=query,
    config={
        "tools": tools
    }
)

response

GenerateContentResponse(
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            function_call=FunctionCall(
              args=<... Max depth ...>,
              id=<... Max depth ...>,
              name=<... Max depth ...>
            ),
            thought_signature=b"\x12\xd9\x02\n\xd6\x02\x01\x11M2\x0f:\xe8\xd2\xf4\x96hG\xa9\xc4\x9c\xef\xbb\x15\xff\xd5v\x13\x95\x066M\xcf{z\xa1\x1f~B\xebvv\xec\xab\xc1\xaf)\x9e\xec3\xb1D\xe8_\xf2\x17|\xf08\x8a\xa5)\xf7\xa0zE\xa0H\xfc':\x12\x96\x92\x805J\x18\xd3\x10B\xd6G` \x1f\x03\xea\xdda\xa6\xcd\x06Y}\xe0P...'
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      index=0
    ),
  ],
  model_version='gemini-3.6-flash',
  response_id='cXCraoPTLduzg8UPq7Tw6Qw',
  sdk_http_response=HttpResponse(
    headers=<dict len=12>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candidates_token_count=21,
    prompt_token_count=352,
    prompt_tokens_

In [21]:
candidate = response.candidates[0]

function_call = candidate.content.parts[0].function_call

print("Tool selected:")
print(function_call.name)

print("\nArguments:")
print(dict(function_call.args))

Tool selected:
get_airport_metrics

Arguments:
{'airport_code': 'SFO'}


## 8. Execute the Selected Tool

Gemini has selected a tool and generated structured arguments.

The application now:

1. Extracts the tool name
2. Extracts the arguments
3. Executes the corresponding Python function
4. Stores the structured tool result

In [25]:
from src.tools import execute_tool

tool_name = function_call.name
tool_arguments = dict(function_call.args)

print("Executing tool:")
print(tool_name)

print("\nArguments:")
print(tool_arguments)

tool_result = execute_tool(
    tool_name,
    tool_arguments
)

print("\nTool result:")
print(tool_result)

Executing tool:
get_airport_metrics

Arguments:
{'airport_code': 'SFO'}

Tool result:
{'status': 'success', 'airport_code': 'SFO', 'completion_rate': 0.86, 'average_eta': 11.2, 'active_drivers': 355, 'driver_cancellation_rate': 0.11, 'queue_size': 260, 'surge_multiplier': 1.4, 'request_volume': 4500, 'timestamp': '2026-09-15 16:00:00'}


## 9. Send Tool Result Back to Gemini

The Python application has executed the tool and received
the operational data.

The tool result is now sent back to Gemini.

Gemini uses the tool result to generate the final answer
for the user.

Flow:

User Query
    ↓
Gemini
    ↓
Function Call
    ↓
Python Tool
    ↓
Tool Result
    ↓
Gemini
    ↓
Final Answer

In [29]:
tool_response_part = types.Part.from_function_response(
    name=tool_name,
    response={
        "result": tool_result
    }
)

print("Tool response created successfully.")

Tool response created successfully.


In [30]:
final_response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=[
        types.Content(
            role="user",
            parts=[
                types.Part.from_text(text=query)
            ]
        ),
        candidate.content,
        types.Content(
            role="user",
            parts=[
                tool_response_part
            ]
        )
    ],
    config={
        "tools": tools
    }
)

print(final_response.text)

Here is the current operational status at **SFO** (as of September 15, 2026, 16:00):

* **Request Volume:** 4,500
* **Active Drivers:** 355
* **Queue Size:** 260
* **Surge Multiplier:** 1.4x
* **Average ETA:** 11.2 minutes
* **Completion Rate:** 86%
* **Driver Cancellation Rate:** 11%
